# RAG Policy Assistant — sandbox/RAG.ipynb

Modular sandbox for building, benchmarking, and iterating on the Meridian Health Group benefits RAG pipeline.  
**All experiments happen here. Nothing moves to `starter/rag_starter.py` until explicitly approved.**

| Step | What you build | Why it matters |
|------|----------------|----------------|
| 1 | Load Documents | Read `data/benefits_policies.md` as raw text |
| 2 | Chunk | Split on `## POL-NNN` boundaries — preserves headers for citation |
| 3 | Embed | `all-MiniLM-L6-v2` — free, local, CPU, ~80 MB one-time download |
| 4 | Retrieve | Cosine similarity over numpy vectors; returns top-k policy chunks |
| 5 | Answer | Claude answers strictly from context, cites POL-NNN, surfaces conflicts |
| 6 | Evaluate | Batch runner over all 15 sample questions |

> Run each cell top-to-bottom with **Shift+Enter**.

---

## Environment Setup

Credentials are loaded from a `.env` file. Required keys: `PORTKEY_API_KEY` and `MY_MODEL`. **Never commit `.env`.**

In [ ]:
%pip install anthropic python-dotenv sentence-transformers numpy -q

### Initialize clients

Anthropic client via Portkey gateway + `sentence-transformers` embedding model loaded once at startup.

In [ ]:
import os
import csv
import re
import numpy as np
from pathlib import Path
from anthropic import Anthropic
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

client = Anthropic(
    api_key=os.environ["PORTKEY_API_KEY"],
    base_url="https://portkeygateway.perficient.com",
    default_headers={"x-portkey-api-key": os.environ["PORTKEY_API_KEY"]}
)
MODEL = os.environ["MY_MODEL"]

EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

DATA_DIR = Path("../data")
POLICIES = DATA_DIR / "benefits_policies.md"
QUESTIONS_CSV = DATA_DIR / "sample_questions.csv"

print(f"Client ready | Model: {MODEL}")
print("Embedding model ready: all-MiniLM-L6-v2")

---
## Step 1: Load Documents

Read `data/benefits_policies.md` as raw text. Maps to `load_documents()` in `rag_starter.py`.

In [ ]:
def load_documents(path=POLICIES):
    text = path.read_text(encoding="utf-8")
    print(f"Loaded {len(text):,} characters")
    print(text[:300])
    return text

text = load_documents()

## Step 2: Chunk

Split the handbook on `## POL-NNN` boundaries so each of the 19 policies becomes exactly one chunk.

**Why this strategy**: policies are short (2–8 lines, well under 200 tokens each), so splitting within a policy
risks orphaning the `POL-NNN` header that `answer()` needs for citation. Compound questions (e.g. HSA+FSA)
are handled by increasing `k` in Step 4 rather than sub-chunking.

Maps to `chunk()` in `rag_starter.py`.

In [ ]:
def chunk(text, target_tokens=200):
    """Split on ## POL-NNN boundaries; each policy becomes one chunk with its header preserved."""
    parts = re.split(r"(?=^## POL-)", text, flags=re.MULTILINE)
    return [p.strip() for p in parts if p.strip().startswith("## POL-")]

chunks = chunk(text)
print(f"{len(chunks)} chunks\n")
for c in chunks[:3]:
    print("---")
    print(c[:200])
    print()

## Step 3: Embed

Convert each chunk to a 384-dimension vector using `sentence-transformers/all-MiniLM-L6-v2`.

**Why this model**: free, runs on CPU, ~80 MB one-time download, good semantic similarity for HR text.
No API calls or keys needed for embedding. Maps to `embed()` in `rag_starter.py`.

In [ ]:
def embed(chunks):
    """Encode all chunks at once; returns float32 numpy array of shape (n_chunks, 384)."""
    vectors = EMBED_MODEL.encode(chunks, convert_to_numpy=True, show_progress_bar=False)
    return vectors.astype("float32")

vectors = embed(chunks)
print(f"Embedding shape: {vectors.shape}")

## Step 4: Retrieve

Embed the question with the same model, compute cosine similarity against all chunk vectors,
return the top-k most similar policy chunks.

**K=3** retrieves enough context for most single-policy questions. Raise to 4–5 for compound
eligibility questions (e.g. HSA+FSA interaction). Maps to `retrieve()` in `rag_starter.py`.

In [ ]:
K = 3  # configurable — raise to 4 or 5 for compound-eligibility questions

def retrieve(question, chunks, vectors, k=K):
    """Cosine similarity retrieval; returns top-k policy chunks."""
    q_vec = EMBED_MODEL.encode([question], convert_to_numpy=True).astype("float32")
    norms = np.linalg.norm(vectors, axis=1)
    q_norm = float(np.linalg.norm(q_vec))
    sims = (vectors @ q_vec.T).squeeze() / (norms * q_norm + 1e-9)
    top_k_idx = np.argsort(sims)[::-1][:k]
    return [chunks[i] for i in top_k_idx]

test_q = "How much does the company match on my 401k and when does it vest?"
retrieved = retrieve(test_q, chunks, vectors)
print(f"Test question: {test_q}\n")
print(f"Retrieved {len(retrieved)} chunks:")
for r in retrieved:
    print(f"  {r.splitlines()[0]}")

## Step 5: Answer

Prompt Claude with the retrieved chunks as the only allowed source of truth.

Hard requirements baked into the system prompt:
- Cite `POL-NNN` ID(s)
- Surface both sides of version conflicts with effective dates (POL-007 vs POL-015)
- Explicitly state internal contradictions (POL-016: 5-yr eligibility vs 7-yr approval)
- Respond "I don't know" when the handbook doesn't cover the question

Maps to `answer()` in `rag_starter.py`.

In [ ]:
SYSTEM_PROMPT = (
    "You are a benefits policy assistant for Meridian Health Group. "
    "The handbook edition date is 1 October 2025 — treat this as today for all date-sensitive questions. "
    "Answer ONLY using the provided policy excerpts. "
    "Always cite the POL-NNN ID(s) you relied on. "
    "If two policies conflict or one supersedes another, surface both and clarify the effective dates — "
    "never silently pick one side. "
    "If a policy contains an internal contradiction, state it explicitly. "
    "If the context does not contain enough information to answer, say exactly: "
    "'I don't know — this is not covered in the handbook.'"
)

def answer(question, retrieved):
    context = "\n\n".join(retrieved)
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=[{
            "role": "user",
            "content": f"Policy excerpts:\n\n{context}\n\n---\n\nQuestion: {question}"
        }]
    )
    return response.content[0].text

print(f"Q: {test_q}\n")
print(answer(test_q, retrieved))

## Step 6: Batch Evaluation

Run all 15 sample questions through the full pipeline. Tricky questions are flagged:
- **Q11** — POL-007 vs POL-015 version conflict (2026 match update)
- **Q12** — POL-016 internal contradiction (5-yr eligibility vs 7-yr approval)
- **Q13** — Unanswerable (PTO accrual during parental leave not in handbook)
- **Q15** — Compound retrieval (POL-005 family HSA seed amount)

In [ ]:
TRICKY = {"Q11", "Q12", "Q13", "Q15"}

with open(QUESTIONS_CSV, newline="", encoding="utf-8") as f:
    questions = [(row["question_id"], row["employee_question"]) for row in csv.DictReader(f)]

for qid, q in questions:
    tag = " [TRICKY]" if qid in TRICKY else ""
    print(f"\n{'='*60}")
    print(f"[{qid}]{tag} {q}")
    print("-" * 60)
    r = retrieve(q, chunks, vectors)
    print(answer(q, r))

---
## Notes & Next Steps

**What to tune:**
- `K` in Step 4 — increase to 4–5 if compound-eligibility questions get incomplete answers
- Chunking granularity — if retrieval misses context, consider splitting long policies into sub-chunks
  while preserving the `## POL-NNN` header prefix in each sub-chunk
- System prompt — tighten citation requirements or add output format constraints

**Once satisfied:** port the five functions to `starter/rag_starter.py`.